# Flow Matching

Flow matching is to say we want a model akin to a continuous normalizing flow, wherein we aim to learn a transformation of a base density with $x_0 \sim p_0$ to a target density $x_{\text{data}} = x_1 \sim p_1$ by way of a solution map to an initial value problem, *but* we want to avoid expensive simulation during training. 

To accomplish this, we first make the observation that any probability path, $p_t: \mathbb{R}^d \times [0,1] \rightarrow \mathbb{R}$ implies a vector field and vice-versa from the continuity equation:

$$ \frac{d}{dt} p_t(x) + \nabla \cdot p_t f_t (x) = 0 $$

Which means that if we can somehow prescribe the probability path we want from $p_0$ to $p_1$, we could then generate some resultant vector field and find an approximation to this vector field.

$$ \mathcal{L}_{FM}(\theta) = \mathbb{E}_{t \sim U(0,1), x_t \sim p_t(x)} \left[||f_\theta(x_t,t) - f(x_t,t)||^2 \right] $$

Unfortunately to *prescribe* a probability path we would actually need to know what our data distribution was, and if we knew that then we wouldn't be fiddling with this flow matching business in the first place. But there is a work around, wherein we instead write out a conditional probability path, which only requires drawing paths between distributions focused on individual data samples and our base distribution. For instance,

$$ 
\begin{aligned}
p_t(x|x_1) &= \mathcal{N}(x|\mu(x_1,t), \sigma^2(x_1,t)\mathbb{I}) \\
&= \mathcal{N}(x| tx_1, (1-t)^2 \mathbb{I})
\end{aligned}
$$

Notice we remain Gaussian throughout the entirety of the path, and that the mean and variance become dead centered at the data sample if $t=1$. This choice of path above (line 2) is linear, and conveniently means that the random variable $x_t = tx_1 + (1-t)x_0 \sim p_t$. From here, it can be deduced that $\frac{d}{dt} x_t = f(x_t, t| x_1) = x_1 - x_0$. Thus we have an analytical form of the ground truth vector field we want to approximate, with the caveat that for this choice of path our base distribution needs to be the standard univariate normal.

$$ \mathcal{L}_{CFM}(\theta) = \mathbb{E}_{t\sim U(0,1), x_t \sim p_t(x|x_1), x_1 \sim p_1(x)} \left[f_\theta(x_t, t) - f(x_t, t; x_1)\right]
$$

And it turns out the optimal $\theta$ for this problem is the same as for the unconditional problem.


As far as the implementation is concerned, we need to (1) collect a a data sample $x_1$ and base sample $x_0$, (2) draw a straight line path between the two, as $x_t = t*x_1 + (1-t)*x_0$, (3) interpret the time derivative as $\frac{d}{dt} x_t = x_1 - x_0$ and set this as our ground truth vector field, for which we want our $f_\theta(x_t,t)$'s output to match over uniform random $t$. 

In [1]:
import orthojax as ojax ### easy orthogonal polynomials
from typing import List, Callable, Tuple
import equinox as eqx
import matplotlib.pyplot as plt
import jax
from jax import random as jr, numpy as jnp
from jaxtyping import Float, Array
from jax.scipy.stats import norm, multivariate_normal
import optax
from layers import NormalizingFlow
from functools import partial
import diffrax
from abc import ABC, abstractmethod

key = jr.PRNGKey(41)

### Choice of Parameterized Vector Field

Here we compose ConcatSquash layers from the [FFJORD](https://arxiv.org/pdf/1810.01367) [codebase](https://github.com/rtqichen/ffjord). 

In [ ]:
class ConcatSquash(eqx.Module):
    lin1: eqx.nn.Linear
    lin2: eqx.nn.Linear
    lin3: eqx.nn.Linear

    def __init__(self, in_size, out_size, *, key):
        keys = jr.split(key, 3)
        self.lin1 = eqx.nn.Linear(in_size, out_size, key=keys[0])
        self.lin2 = eqx.nn.Linear(1, out_size, key=keys[1])
        self.lin3 = eqx.nn.Linear(1, out_size, use_bias=False, key=keys[2])

    def __call__(self, t, x):
        return self.lin1(x) * jax.nn.sigmoid(self.lin2(t)) + self.lin3(t)

### Compositional
class VectorField(eqx.Module):
    layers: List[eqx.Module]
    
    def __init__(self, base_layer, data_size, width_size, depth, *, key):
        layers = []
        keys = jr.split(key, depth+1)
        if depth == 0: layers.append(base_layer(in_size=data_size, out_size=data_size, key=keys[0]))
        else:
            layers.append(base_layer(in_size=width_size, out_size=width_size, key=keys[0]))
            [layers.append(base_layer(in_size=width_size, out_size=width_size, key=k)) for k in keys[1:-1]]
            layers.append(base_layer(in_size=width_size, out_size=data_size, key=keys[-1]))
        self.layers = layers

    def __call__(self, t, x, args):
        t = jnp.asarray(t)[None] ### make t shape (1,) vs scalar
        for layer in self.layers[:-1]:
            x = layer(t, x)
            x = jax.nn.tanh(x)
        x = self.layers[-1](t, x)
        return x

### Model 

The model has two functions, one which computes the loss between vector fields for training given a data sample x, the other which takes a random key to generate a sample the base distribution and thereafter solve the ode with our parameterized vector field forward in time to morph that sample to one which matches the likeness of the data. 

In [ ]:
class Flow(eqx.Module):
    base_dist: Callable
    Func: eqx.Module
    def __init__(self, *, vf, base_dist, key):
        self.Func = vf(key=key)
        self.base_dist = base_dist
     
    def train(self, x, key):
        x_1 = x
        x_0 = self.base_dist.rvs(key=key, shape=())
        ground_truth_dxdt = x_1 - x_0 ### x_dim

        key,_ = jr.split(key)
        t = jr.uniform(key)
        x_t = x_0 * (1 - t) + x_1 * t
        pred_dxdt = self.Func(t,x_t,args=None)
        return ((ground_truth_dxdt - pred_dxdt) ** 2).sum(axis=-1) 
        
    def sample(self, key):
        x = self.base_dist.rvs(key=key, shape=())
        term = diffrax.ODETerm(self.Func)
        solver = diffrax.Tsit5()
        sol = diffrax.diffeqsolve(term, solver, 0, 1, 0.05, x)
        (x,) = sol.ys
        return x

### Problem setup

We aim to learn a correlated Gaussian distribution from a standard multivariate normal.

In [8]:
x_dim = 4
keys = jr.split(key, 3)

n_samples = 1000

n_tri = int(x_dim*(x_dim+1)/2)
L = jr.uniform(keys[0], (n_tri,))
L = jnp.zeros((x_dim,x_dim)).at[jnp.tril_indices(x_dim)].set(L)
Sigma = L @ L.T ### valid covariance

mu = jr.uniform(keys[1], (x_dim,))
x_true_mu, x_true_cov = mu, Sigma
x_true_samples = jr.multivariate_normal(keys[2], mean=mu, cov=Sigma, shape=(n_samples,))

In [9]:
class FixedStandardNormal(eqx.Module):
    dim: int
    def __init__(self, *, dim,):
        self.dim = dim
    def logpdf(self, x):
        return multivariate_normal.logpdf(x, mean=jnp.zeros((self.dim,)), cov=jnp.eye(self.dim))
    def rvs(self, key, shape):
        return jr.multivariate_normal(key, mean=jnp.zeros((self.dim,)), cov=jnp.eye(self.dim), shape=shape)

### could swap this to polynomial or kernel or whatever we want
base_layer = ConcatSquash
Func = partial(VectorField, base_layer=base_layer, data_size=x_dim, width_size=x_dim, depth=3)
model = Flow(vf=Func, 
             base_dist=TrainableStandardNormal(dim=x_dim, key=key),
             key=key)

In [10]:
epochs = 10000
opt = optax.adamw(1e-3, weight_decay=0.1)
opt_state = opt.init(eqx.filter(model, eqx.is_inexact_array))

In [11]:
@eqx.filter_jit
def train_step(model, opt_state, batch, batch_key):
    x = batch
    keys = jr.split(batch_key, len(x))
    def vf_loss(model):
        loss = eqx.filter_vmap(model.train, in_axes=(eqx.if_array(0),0))(x, keys)
        return jnp.mean(loss)

    loss, grads = eqx.filter_value_and_grad(vf_loss)(model)
    updates, opt_state = opt.update(grads, opt_state, eqx.filter(model, eqx.is_inexact_array))
    model = eqx.apply_updates(model, updates)
    return model, opt_state, loss 

@eqx.filter_jit
def eval(model, key):
    keys = jr.split(key, 1000)
    x_pred_samples = jax.vmap(model.sample)(keys)
    x_pred_mu = jnp.mean(x_pred_samples, axis=0)
    x_pred_cov = jnp.cov(x_pred_samples, rowvar=False)
    mu_loss, std_loss = jnp.linalg.norm(x_pred_mu - x_true_mu) / jnp.linalg.norm(x_true_mu), jnp.linalg.norm(x_pred_cov - x_true_cov)/jnp.linalg.norm(x_true_cov)
    return mu_loss, std_loss

### Loop

In [12]:
mu_losses, std_losses = [],[]
for epoch in range(10000):
    key, epoch_key = jr.split(key)
    model, opt_state, loss = train_step(model, opt_state, x_true_samples, epoch_key)
    mu_loss, std_loss = eval(model, epoch_key)
    mu_losses.append(mu_loss), std_losses.append(std_loss)
    if epoch % 100 == 0:
        print(f'{epoch=}, nll: {loss.item():.5f}, mu_loss: {mu_loss.item():.5f}, std_loss: {std_loss.item():.5f}') 

epoch=0, nll: 6.78683, mu_loss: 1.12578, std_loss: 0.99546
epoch=100, nll: 5.37291, mu_loss: 0.72612, std_loss: 0.99386
epoch=200, nll: 4.80151, mu_loss: 0.40983, std_loss: 0.89230
epoch=300, nll: 4.19211, mu_loss: 0.20704, std_loss: 0.78334
epoch=400, nll: 3.93924, mu_loss: 0.12043, std_loss: 0.63566
epoch=500, nll: 3.67770, mu_loss: 0.08436, std_loss: 0.59498
epoch=600, nll: 3.27235, mu_loss: 0.07830, std_loss: 0.60900
epoch=700, nll: 3.24856, mu_loss: 0.06482, std_loss: 0.60615
epoch=800, nll: 3.13601, mu_loss: 0.05735, std_loss: 0.59457
epoch=900, nll: 2.90206, mu_loss: 0.03743, std_loss: 0.61098
epoch=1000, nll: 2.79929, mu_loss: 0.04993, std_loss: 0.47654
epoch=1100, nll: 2.61687, mu_loss: 0.05145, std_loss: 0.35598
epoch=1200, nll: 2.67581, mu_loss: 0.09829, std_loss: 0.31938
epoch=1300, nll: 2.39045, mu_loss: 0.05367, std_loss: 0.26014
epoch=1400, nll: 2.25182, mu_loss: 0.04693, std_loss: 0.23634
epoch=1500, nll: 2.26016, mu_loss: 0.03203, std_loss: 0.19095
epoch=1600, nll: 2.0

KeyboardInterrupt: 